# 判案 API · Colab 部署

**跑之前要做两件事：**
1. 把 `judgments.jsonl`（31 MB）传到 Google Drive 根目录，路径为 `MyDrive/judgments.jsonl`
2. 在侧边栏 🔑（Secrets）里加一条 `DEEPSEEK_KEY`，值是你的 DeepSeek API key，并打开 "Notebook access"

**然后从上到下运行单元格。** 最后一个会打印 `https://xxxx.trycloudflare.com`，拿它去提交。

> ⚠️ Colab 免费版闲置约 90 分钟会被回收，最长约 12 小时。**会话一断，网址就变了，必须重跑并重新提交。**
> 老师测试期间请保持这个标签页开着，别关。

In [ ]:
# 1) 挂载 Google Drive（用来读取判决书）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) 拉代码（只拉代码，判决书不进仓库）
!rm -rf /content/case-judge-api
!git clone --depth 1 https://github.com/wzilu-star/-4.git /content/case-judge-api
!ls /content/case-judge-api

In [ ]:
# 3) 把判决书从 Drive 拷进来
import os, shutil
os.makedirs('/content/case-judge-api/judge_data', exist_ok=True)
os.makedirs('/content/case-judge-api/logs', exist_ok=True)
SRC = '/content/drive/MyDrive/judgments.jsonl'
DST = '/content/case-judge-api/judge_data/judgments.jsonl'
shutil.copy(SRC, DST)
print('OK', os.path.getsize(DST) / 1e6, 'MB')

In [ ]:
# 4) 写入 .env（API key）与大模型环境变量
import os
from google.colab import userdata

JUDGE_KEY = 'sugu-2f7c9a41d6b3'          # 这个就是交给老师的 key
with open('/content/case-judge-api/.env', 'w') as f:
    f.write('JUDGE_API_KEY=%s\n' % JUDGE_KEY)

os.environ['ANTHROPIC_BASE_URL'] = 'https://api.deepseek.com/anthropic'
os.environ['ANTHROPIC_MODEL'] = 'deepseek-flash'
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('DEEPSEEK_KEY')
except Exception:
    import getpass
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('DeepSeek API key: ')
print('key 前缀:', os.environ['ANTHROPIC_API_KEY'][:6], '...')

In [ ]:
# 5) 一键：建库 → 起 API → 开隧道 → 打印公网地址
#    首次运行约 4-6 分钟（要建索引 + 补标案由）
%cd /content/case-judge-api
!JUDGMENTS=/content/case-judge-api/judge_data/judgments.jsonl bash colab/run_colab.sh

In [ ]:
# 6) 自测：把上面打印的地址填进来
URL = 'https://xxxx.trycloudflare.com'   # ← 换成上面打印的地址
!python -I tools/smoke_api.py --url {URL} --limit 3

## 提交内容（三行）

```
API 地址：https://xxxx.trycloudflare.com
Key：sugu-2f7c9a41d6b3
方法：把 3,520 份判决书按「本院认为」切成案情与判决结果，离线抽出案由/法条/方向/主文做结构化标签，服务时用 BM25 按案情检索同类已判案例，由 DeepSeek 参照这些类案按本案事实生成四项结果。
```